# Notebook 19 — Generic HeavyHex Circuit Handler

**Purpose:**  
Provided circuits for multiple distances (d=5 now, d=7 and d=9 coming).  
This notebook auto-detects the structure of ANY circuit provided and builds  
the correct input tensors for all decoder types — no hardcoding required.

**How it works:**  
Instead of assuming `12 + 9×24 + 12 = 240`, it inspects the circuit's detector  
blocks, finds the initial/per-round/final structure automatically, and reshapes  
accordingly. Works for any distance and any number of rounds.

**Circuits provided:**
```
s_stim_circ_z.pickle   → d=5, Standard Z basis, 9 rounds  (already have this)
s_stim_circ_x.pickle   → d=5, Standard X basis, 9 rounds  (already have this)
s_stim_circ_z_d7.pickle → d=7, Standard Z basis           (don't have this currently)
s_stim_circ_z_d9.pickle → d=9, Standard Z basis, richer noise (don't have this currently)
```

**Prerequisite:** None — this notebook is standalone.  
```bash
pip install stim pymatching numpy
```

In [1]:
import numpy as np
import stim
import pymatching
import pickle
import os
from typing import Dict, Tuple, Optional
from dataclasses import dataclass

print(f"Stim       : {stim.__version__}")
print(f"PyMatching : {pymatching.__version__}")

Stim       : 1.15.0
PyMatching : 2.3.1


---
## 1. Circuit Inspector

Auto-detects the detector block structure from any Stim circuit.

In [2]:
def load_circuit_from_pickle(pkl_path: str):
    """Load a stim.Circuit from pickle — bypasses stim version mismatch."""
    with open(pkl_path, "rb") as f:
        raw = f.read()
    start   = raw.find(b"QUBIT_COORDS")
    obs_pos = raw.rfind(b"OBSERVABLE_INCLUDE")
    if start == -1 or obs_pos == -1:
        raise ValueError(f"Not a valid stim circuit pickle: {pkl_path}")
    end = obs_pos
    while end < len(raw):
        b = raw[end]
        if b == 0x0A: end += 1; break
        elif b > 0x7E or (b < 0x20 and b not in (0x09, 0x0D)): break
        end += 1
    return stim.Circuit(raw[start:end].decode("ascii"))

@dataclass
class CircuitStructure:
    """
    Describes the detector block structure of a Stim circuit.

    For Standard HeavyHex circuits:
      n_initial  = 12  (prep round)
      n_per_round = 24  (syndrome rounds)
      n_rounds    = 9
      n_final    = 12  (data qubit readout)
      total      = 240
    """
    n_initial:    int    # detectors in first block
    n_per_round:  int    # detectors in each middle block
    n_rounds:     int    # number of syndrome rounds
    n_final:      int    # detectors in last block
    total_det:    int    # total detectors
    n_qubits:     int    # physical qubits
    n_meas:       int    # total measurements per shot
    step_width:   int    # padded width for temporal input (max of blocks)
    n_steps:      int    # total time steps for temporal input


def inspect_circuit(circuit: stim.Circuit) -> CircuitStructure:
    """
    Auto-detect the detector block structure of a HeavyHex circuit.

    Parses the circuit as text to find detector blocks, since the
    Stim Python API does not directly expose per-block detector counts.
    """
    circ_str = str(circuit)
    lines = circ_str.strip().split('\n')

    # Find detector blocks (contiguous groups of DETECTOR lines)
    blocks = []
    i = 0
    while i < len(lines):
        if lines[i].strip().startswith('DETECTOR'):
            count = 0
            while i < len(lines) and lines[i].strip().startswith('DETECTOR'):
                count += 1
                i += 1
            blocks.append(count)
        else:
            i += 1

    if len(blocks) < 3:
        raise ValueError(f"Expected at least 3 detector blocks, found {len(blocks)}")

    n_initial   = blocks[0]
    n_final     = blocks[-1]
    middle      = blocks[1:-1]

    # All middle blocks should be the same size
    if len(set(middle)) != 1:
        raise ValueError(f"Middle blocks are not uniform: {middle}")

    n_per_round = middle[0]
    n_rounds    = len(middle)
    total_det   = circuit.num_detectors

    # Padded step width = max of any block (for LSTM/Transformer)
    step_width = max(n_initial, n_per_round, n_final)
    n_steps    = 1 + n_rounds + 1   # initial + rounds + final

    return CircuitStructure(
        n_initial=n_initial,
        n_per_round=n_per_round,
        n_rounds=n_rounds,
        n_final=n_final,
        total_det=total_det,
        n_qubits=circuit.num_qubits,
        n_meas=circuit.num_measurements,
        step_width=step_width,
        n_steps=n_steps,
    )


# Test on the circuit we have
circuit_z = load_circuit_from_pickle("s_stim_circ_z.pickle")

struct = inspect_circuit(circuit_z)
print("s_stim_circ_z structure:")
print(f"  Physical qubits  : {struct.n_qubits}")
print(f"  Measurements/shot: {struct.n_meas}")
print(f"  Total detectors  : {struct.total_det}")
print(f"  Initial block    : {struct.n_initial} detectors")
print(f"  Per-round block  : {struct.n_per_round} detectors × {struct.n_rounds} rounds")
print(f"  Final block      : {struct.n_final} detectors")
print(f"  Temporal shape   : ({struct.n_steps}, {struct.step_width})  for LSTM/Transformer")
print()
expected = struct.n_initial + struct.n_per_round * struct.n_rounds + struct.n_final
assert expected == struct.total_det, f"Mismatch: {expected} != {struct.total_det}"
print(f"  Structure verified: {struct.n_initial} + {struct.n_rounds}×{struct.n_per_round} + {struct.n_final} = {struct.total_det} ✓")

s_stim_circ_z structure:
  Physical qubits  : 135
  Measurements/shot: 425
  Total detectors  : 240
  Initial block    : 12 detectors
  Per-round block  : 24 detectors × 9 rounds
  Final block      : 12 detectors
  Temporal shape   : (11, 24)  for LSTM/Transformer

  Structure verified: 12 + 9×24 + 12 = 240 ✓


---
## 2. Generic Data Sampler

Works for any circuit, auto-detecting the temporal structure.

In [3]:
def sample_circuit(circuit: stim.Circuit,
                   num_shots: int,
                   struct: Optional[CircuitStructure] = None
                   ) -> Dict[str, np.ndarray]:
    """
    Sample from a circuit and return all input formats needed by decoders.

    Returns dict with keys:
      'det_flat'    : (N, total_det)          float32  MLP input
      'det_temporal': (N, n_steps, step_width) float32  LSTM/Transformer input
      'raw_meas'    : (N, n_meas)              float32  raw measurement bits
      'obs'         : (N,)                     float32  logical observable
      'struct'      : CircuitStructure object
    """
    if struct is None:
        struct = inspect_circuit(circuit)

    # Sample raw measurements
    sampler  = circuit.compile_sampler()
    raw_all  = np.array(sampler.sample(shots=num_shots), dtype=np.bool_)

    # Convert to detection events via m2d
    converter        = circuit.compile_m2d_converter()
    det_events, obs  = converter.convert(
        measurements=raw_all, separate_observables=True)
    det_events = np.array(det_events, dtype=np.float32)
    obs        = np.array(obs, dtype=np.float32).squeeze()

    # Build temporal input: (N, n_steps, step_width)
    N  = num_shots
    sw = struct.step_width
    ns = struct.n_steps
    temporal = np.zeros((N, ns, sw), dtype=np.float32)

    # Step 0: initial block (padded to step_width)
    ni = struct.n_initial
    temporal[:, 0, :ni] = det_events[:, :ni]

    # Steps 1 to n_rounds: per-round blocks
    nr  = struct.n_per_round
    off = ni
    for r in range(struct.n_rounds):
        temporal[:, r+1, :nr] = det_events[:, off : off+nr]
        off += nr

    # Last step: final block (padded to step_width)
    nf = struct.n_final
    temporal[:, -1, :nf] = det_events[:, off : off+nf]

    return {
        'det_flat':     det_events,
        'det_temporal': temporal,
        'raw_meas':     raw_all.astype(np.float32),
        'obs':          obs,
        'struct':       struct,
    }


def run_mwpm(circuit: stim.Circuit,
             det_events: np.ndarray,
             obs: np.ndarray) -> float:
    """
    Run MWPM and return logical error rate.
    Uses approximate_disjoint_errors=True as required for ACES noise.
    """
    dem = circuit.detector_error_model(
        decompose_errors=True,
        approximate_disjoint_errors=True)
    matching  = pymatching.Matching.from_detector_error_model(dem)
    predicted = matching.decode_batch(det_events.astype(np.bool_))
    return float(np.any(predicted != obs[:, None].astype(np.bool_), axis=1).mean())


print("Sampling functions defined.")

Sampling functions defined.


---
## 3. Load & Process All Available Circuits

In [4]:
NUM_SHOTS = 100_000

# Dictionary of circuits to process.
# Add new entries here as more circuits are provided.
CIRCUITS = {
    "s_stim_circ_z": "s_stim_circ_z.pickle",   # d=5, Standard Z
    "s_stim_circ_x": "s_stim_circ_x.pickle",   # d=5, Standard X
    # Add when provided them:
    # "s_stim_circ_z_d7": "s_stim_circ_z_d7.pickle",  # d=7
    # "s_stim_circ_z_d9": "s_stim_circ_z_d9.pickle",  # d=9
}

all_results = {}

for name, pkl_file in CIRCUITS.items():
    if not os.path.exists(pkl_file):
        print(f"SKIP: {pkl_file} not found")
        continue

    print(f"\n{'='*55}")
    print(f"Processing: {name}")
    print('='*55)

    # load_circuit_from_pickle is defined in cell 3
    circuit = load_circuit_from_pickle(pkl_file)

    # Auto-detect structure
    struct = inspect_circuit(circuit)
    print(f"  Qubits={struct.n_qubits}  "
          f"Det={struct.total_det}  "
          f"Structure: {struct.n_initial}+{struct.n_rounds}\u00d7{struct.n_per_round}+{struct.n_final}")
    print(f"  LSTM/Transformer input: ({struct.n_steps}, {struct.step_width})")

    # Sample
    print(f"  Sampling {NUM_SHOTS:,} shots...")
    data = sample_circuit(circuit, NUM_SHOTS, struct)

    trivial_ler = float(data['obs'].mean())
    print(f"  Trivial LER: {trivial_ler:.4f}  ({100*trivial_ler:.3f}%)")

    # MWPM
    print(f"  Running MWPM...")
    mwpm_ler = run_mwpm(circuit, data['det_flat'], data['obs'])
    print(f"  MWPM LER: {mwpm_ler:.4f}  ({trivial_ler/mwpm_ler:.1f}x suppression)")

    # Save data
    save_dir = f"data/heavyhex_{name}"
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(f"results/heavyhex_{name}", exist_ok=True)

    np.save(f"{save_dir}/detection_events.npy",  data['det_flat'])
    np.save(f"{save_dir}/det_temporal.npy",       data['det_temporal'])
    np.save(f"{save_dir}/raw_measurements.npy",   data['raw_meas'])
    np.save(f"{save_dir}/observable_flips.npy",   data['obs'])
    np.save(f"results/heavyhex_{name}/mwpm_result.npy",
            np.array([mwpm_ler, trivial_ler]))

    # Save structure info
    import json as _json
    with open(f"{save_dir}/structure.json", "w") as f:
        _json.dump({
            'n_initial':   struct.n_initial,
            'n_per_round': struct.n_per_round,
            'n_rounds':    struct.n_rounds,
            'n_final':     struct.n_final,
            'total_det':   struct.total_det,
            'n_qubits':    struct.n_qubits,
            'n_meas':      struct.n_meas,
            'step_width':  struct.step_width,
            'n_steps':     struct.n_steps,
        }, f, indent=2)

    print(f"  Saved to {save_dir}/")

    all_results[name] = {
        'struct':      struct,
        'trivial_ler': trivial_ler,
        'mwpm_ler':    mwpm_ler,
        'data':        data,
    }

print("\nAll circuits processed.")



Processing: s_stim_circ_z
  Qubits=135  Det=240  Structure: 12+9×24+12
  LSTM/Transformer input: (11, 24)
  Sampling 100,000 shots...
  Trivial LER: 0.4990  (49.903%)
  Running MWPM...
  MWPM LER: 0.2256  (2.2x suppression)
  Saved to data/heavyhex_s_stim_circ_z/

Processing: s_stim_circ_x
  Qubits=135  Det=240  Structure: 12+9×24+12
  LSTM/Transformer input: (11, 24)
  Sampling 100,000 shots...
  Trivial LER: 0.4990  (49.899%)
  Running MWPM...
  MWPM LER: 0.2271  (2.2x suppression)
  Saved to data/heavyhex_s_stim_circ_x/

All circuits processed.


---
## 4. Cross-Circuit Summary

In [5]:
print("=" * 70)
print("Summary across all processed circuits")
print("=" * 70)
print(f"  {'Circuit':<22}  {'Qubits':>7}  {'Det':>5}  "
      f"{'Shape':>12}  {'Trivial':>8}  {'MWPM':>8}  {'Supp.':>6}")
print("  " + "-"*72)

for name, res in all_results.items():
    s = res['struct']
    shape_str = f"({s.n_steps},{s.step_width})"
    supp = res['trivial_ler'] / res['mwpm_ler'] if res['mwpm_ler'] > 0 else float('inf')
    print(f"  {name:<22}  {s.n_qubits:>7}  {s.total_det:>5}  "
          f"{shape_str:>12}  "
          f"{100*res['trivial_ler']:>7.3f}%  "
          f"{100*res['mwpm_ler']:>7.3f}%  "
          f"{supp:>5.1f}x")

print()
print("The 'Shape' column = (n_steps, step_width) for LSTM/Transformer input.")
print("When Robin provides d=7/d=9 circuits, add them to CIRCUITS dict above")
print("and re-run — everything adapts automatically.")

Summary across all processed circuits
  Circuit                  Qubits    Det         Shape   Trivial      MWPM   Supp.
  ------------------------------------------------------------------------
  s_stim_circ_z               135    240       (11,24)   49.903%   22.561%    2.2x
  s_stim_circ_x               135    240       (11,24)   49.899%   22.710%    2.2x

The 'Shape' column = (n_steps, step_width) for LSTM/Transformer input.
When Robin provides d=7/d=9 circuits, add them to CIRCUITS dict above
and re-run — everything adapts automatically.


---
## 5. Generic Decoder Training Function

Takes any circuit's data (from `all_results`) and trains all four decoder types.  
Reusable for d=7, d=9 as and when provided with them.

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42


def make_decoder_for_structure(arch: str, struct: CircuitStructure) -> nn.Module:
    """
    Build the right decoder architecture for a given circuit structure.
    Scales hidden dimensions automatically based on the input size.
    """
    if arch == 'mlp':
        input_dim = struct.total_det
        h = max(256, input_dim * 2)
        layers, in_d = [], input_dim
        for size in [h, h//2, h//4]:
            layers += [nn.Linear(in_d,size), nn.BatchNorm1d(size), nn.ReLU(), nn.Dropout(0.3)]
            in_d = size
        layers.append(nn.Linear(in_d, 1))
        class MLP(nn.Module):
            def __init__(self):
                super().__init__(); self.net = nn.Sequential(*layers)
            def forward(self, x): return self.net(x).squeeze(-1)
        return MLP()

    elif arch == 'lstm':
        input_size  = struct.step_width
        lstm_hidden = max(64, input_size * 3)
        class LSTM(nn.Module):
            def __init__(self):
                super().__init__()
                self.lstm = nn.LSTM(input_size, lstm_hidden, 2,
                                    batch_first=True, dropout=0.2)
                h = lstm_hidden
                self.head = nn.Sequential(
                    nn.Linear(h, h//2), nn.ReLU(), nn.Dropout(0.2),
                    nn.Linear(h//2, h//4), nn.ReLU(), nn.Dropout(0.2),
                    nn.Linear(h//4, 1))
            def forward(self, x):
                _, (h_n, _) = self.lstm(x)
                return self.head(h_n[-1]).squeeze(-1)
        return LSTM()

    elif arch == 'transformer':
        input_size = struct.step_width
        d_model    = max(64, (input_size // 8 + 1) * 8)  # round up to multiple of 8 for nhead=4
        class Transformer(nn.Module):
            def __init__(self):
                super().__init__()
                self.proj = nn.Linear(input_size, d_model)
                self.pos  = nn.Embedding(struct.n_steps, d_model)
                enc = nn.TransformerEncoderLayer(
                    d_model=d_model, nhead=4, dim_feedforward=d_model*2,
                    dropout=0.1, batch_first=True)
                self.tf   = nn.TransformerEncoder(enc, num_layers=2)
                self.head = nn.Sequential(
                    nn.Linear(d_model, 32), nn.ReLU(), nn.Dropout(0.1),
                    nn.Linear(32, 1))
            def forward(self, x):
                t = self.proj(x) + self.pos(
                    torch.arange(x.shape[1], device=x.device)).unsqueeze(0)
                return self.head(self.tf(t).mean(1)).squeeze(-1)
        return Transformer()

    else:
        raise ValueError(f"Unknown arch: {arch}")


def train_decoders_for_circuit(circuit_name: str,
                                data: dict,
                                struct: CircuitStructure,
                                mwpm_ler: float,
                                epochs: int = 100,
                                patience: int = 15):
    """
    Train all decoder architectures for one circuit.
    Works for any distance/structure auto-detected by inspect_circuit().
    """
    torch.manual_seed(SEED); np.random.seed(SEED)

    obs = data['obs']
    idx = np.arange(len(obs))
    idx_tr, idx_tmp = train_test_split(idx, test_size=0.30,
                                        stratify=obs.astype(int), random_state=SEED)
    idx_val, idx_te = train_test_split(idx_tmp, test_size=0.50,
                                        stratify=obs[idx_tmp].astype(int), random_state=SEED)

    obs_tr  = obs[idx_tr];  obs_val = obs[idx_val];  obs_te = obs[idx_te]
    trivial = obs_te.mean()

    results = {}

    arch_configs = [
        ('mlp',         data['det_flat']),
        ('lstm',        data['det_temporal']),
        ('transformer', data['det_temporal']),
    ]

    for arch, X in arch_configs:
        print(f"\n  Training {arch.upper()} for {circuit_name}...")
        model = make_decoder_for_structure(arch, struct).to(DEVICE)
        params = sum(p.numel() for p in model.parameters())
        print(f"    Parameters: {params:,}")

        X_tr, X_val, X_te = X[idx_tr], X[idx_val], X[idx_te]

        opt  = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        sch  = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=6, factor=0.5)
        pw   = torch.tensor([(1-obs_tr.mean())/max(obs_tr.mean(),1e-6)]).to(DEVICE)
        crit = nn.BCEWithLogitsLoss(pos_weight=pw)

        tr_l  = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(obs_tr)),
                           batch_size=512, shuffle=True)
        va_l  = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(obs_val)),
                           batch_size=1024)

        best_vl, best_ep, best_st = np.inf, 0, None
        t0 = time.time()

        for ep in range(1, epochs+1):
            model.train()
            for Xb, yb in tr_l:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                opt.zero_grad()
                loss = crit(model(Xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            model.eval()
            vl, vc, vt = 0.0, 0, 0
            with torch.no_grad():
                for Xb, yb in va_l:
                    Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                    logits = model(Xb)
                    vl += crit(logits, yb).item()*len(yb)
                    vc += ((torch.sigmoid(logits)>0.5).float()==yb).sum().item()
                    vt += len(yb)
            vl /= vt; sch.step(vl)
            if vl < best_vl:
                best_vl=vl; best_ep=ep
                best_st={k:v.cpu().clone() for k,v in model.state_dict().items()}
            if ep-best_ep >= patience: break

        model.load_state_dict(best_st)
        model.eval()

        with torch.no_grad():
            te_t  = torch.tensor(X_te)
            preds = []
            for i in range(0, len(te_t), 2048):
                preds.append((torch.sigmoid(model(te_t[i:i+2048].to(DEVICE)))>0.5).cpu().numpy())
            pred = np.concatenate(preds)

        ler = float((pred != obs_te).mean())
        print(f"    Test LER: {ler:.5f}  ({trivial/ler:.1f}x vs trivial)  [{time.time()-t0:.0f}s]")

        results[arch] = {'ler': ler, 'params': params, 'model': model}

        os.makedirs(f"models/heavyhex_{circuit_name}", exist_ok=True)
        torch.save({'model_state': model.state_dict(), 'test_ler': ler,
                    'arch': arch, 'circuit': circuit_name},
                   f"models/heavyhex_{circuit_name}/{arch}.pt")

    return results, trivial


print("Generic decoder training function ready.")
print()
print("Example usage:")
print("  results, trivial = train_decoders_for_circuit(")
print("      's_stim_circ_z', all_results['s_stim_circ_z']['data'],")
print("      all_results['s_stim_circ_z']['struct'],")
print("      all_results['s_stim_circ_z']['mwpm_ler'])")

Generic decoder training function ready.

Example usage:
  results, trivial = train_decoders_for_circuit(
      's_stim_circ_z', all_results['s_stim_circ_z']['data'],
      all_results['s_stim_circ_z']['struct'],
      all_results['s_stim_circ_z']['mwpm_ler'])


---
## 6. Train Decoders for s_stim_circ_z

Run this cell. When provided d=7/d=9, duplicate the call with the new circuit name.

In [7]:
circuit_name = "s_stim_circ_z"
res = all_results.get(circuit_name)

if res is None:
    print(f"{circuit_name} was not loaded — check the pickle file exists.")
else:
    nn_results, trivial = train_decoders_for_circuit(
        circuit_name,
        res['data'],
        res['struct'],
        res['mwpm_ler'],
    )

    print()
    print("=" * 58)
    print(f"Results for {circuit_name}")
    print("=" * 58)
    print(f"  {'Decoder':<16}  {'LER':>9}  {'vs Trivial':>11}  {'Params':>10}")
    print("  " + "-"*50)
    print(f"  {'Trivial':<16}  {100*trivial:>8.4f}%  {'—':>11}  {'—':>10}")
    for arch, r in nn_results.items():
        print(f"  {arch.upper():<16}  {100*r['ler']:>8.4f}%  "
              f"{trivial/r['ler']:>9.1f}x  {r['params']:>10,}")
    print(f"  {'MWPM':<16}  {100*res['mwpm_ler']:>8.4f}%  "
          f"{trivial/res['mwpm_ler']:>9.1f}x  {'—':>10}")


  Training MLP for s_stim_circ_z...
    Parameters: 261,841
    Test LER: 0.49893  (1.0x vs trivial)  [16s]

  Training LSTM for s_stim_circ_z...
    Parameters: 73,585
    Test LER: 0.50133  (1.0x vs trivial)  [23s]

  Training TRANSFORMER for s_stim_circ_z...
    Parameters: 71,361
    Test LER: 0.50147  (1.0x vs trivial)  [34s]

Results for s_stim_circ_z
  Decoder                 LER   vs Trivial      Params
  --------------------------------------------------
  Trivial            49.9000%            —           —
  MLP                49.8933%        1.0x     261,841
  LSTM               50.1333%        1.0x      73,585
  TRANSFORMER        50.1467%        1.0x      71,361
  MWPM               22.5610%        2.2x           —
